# 8.1 Merging ticker changes
*(For myself I skip this part. Renaming give to much headache, even though it's not that important for short-term signals. Also it does not affect the price itself.)*

This is optional. If you want it ticker-centric or don't want to get a stockanalysis.com subscription, you can just skip this part.

To get a list of ticker changes, We can loop through all tickers and query <code>Ticker Events</code> but this only works with non-delisted companies. And although you can infer it based on the ticker list by looking at whether the cik or figi has changed, that is very messy. Because a company can stay the same even if the ticker and cik/figi change. I actually did it, and it did found that it did not match the Polygon <code>Ticker Events</code>. Then I stumbled on [stockanalysis.com](https://stockanalysis.com/actions/changes/) where you can find all ticker changes for only 10 bucks a month. The first month is even free. You have to manually download them for each year and put them in the <code>stockanalysis/raw/ticker_changes/</code> folder.

After merging those we will save the result to <code>raw/renamings.csv</code> which will contain the columns <code>['from', 'to', 'now', 'date']</code>.

In [1]:
from tickers import get_tickers, get_ticker_changes
from times import get_market_dates, get_market_calendar, last_trading_date_before
from datetime import datetime, date, time
from pathlib import Path
import pandas as pd
import shutil
import os
DATA_PATH = "../data/polygon/"
# END_DATE = last_trading_date_before(date(2026, 6, 4)) # till today only or else you will get "out of range" and don't waste loop cycles
# END_DATE will need to support updating everyday so that we only fetch the new adjustments in an idempotent way


In [2]:
# This can be done once and then updates can be done with manually appending the list of ticker changes.
###
# Aggregate the csv's
all_ticker_changes = []
for file in os.listdir(DATA_PATH + "../stockanalysis/raw/ticker_changes/"):
    ticker_changes_year = pd.read_csv(DATA_PATH + "../stockanalysis/raw/ticker_changes/" + file, \
        parse_dates=True, index_col=0, usecols=["Date", "Old", "New"])
    all_ticker_changes.append(ticker_changes_year)

ticker_changes = pd.concat(all_ticker_changes)
ticker_changes.rename(columns={"Old": "from", 
                               "New": "to"}, inplace=True)
ticker_changes.index.names = ['date']
ticker_changes.sort_index(inplace=True)
ticker_changes.to_csv(DATA_PATH + "../stockanalysis/ticker_changes.csv")

In [3]:
ticker_changes = pd.read_csv(DATA_PATH + "../stockanalysis/ticker_changes.csv")
print(len(ticker_changes))
ticker_changes[ticker_changes['from'] == "FB"]

5897


,date,from,to
4858,2022-06-09,FB,META


# More problems

Somes there are special conditions, such as 'delinquent', which adds an extra letter at the end of the ticker. E.g. AAII went delinquent and then the ticker became AAIIE. However these are not real ticker changes so it is not contained in the stockanalysis database. However we can easily infer it from our own ticker list: if the dates are consecutive and an extra letter is added, we can infer the ticker change. We will save this to <code>raw/inferred_renamings.csv.</code>

In [4]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH = "../data/polygon/"
RAW       = DATA_PATH + "raw"
PROCESSED = DATA_PATH + "processed"

# ── Status suffix allow-list ──────────────────────────────────────────────────
# Restricting to exchange-assigned status letters eliminates false positives
# (two unrelated companies that happen to satisfy the date+prefix rule).
#   E  SEC-reporting delinquent
#   Q  bankruptcy / liquidation proceedings
#   D  new issue (when-issued / distribution)
# https://chatgpt.com/share/6a327201-6d30-83eb-977a-f0cae5c483ea - no need for other Suffixes like D or V since those have false positives and I want "Precision over Recall" - The Quant Golden Rule. 
# And besides, if a ticker gets to that stage, it's not gonnamake our scanner. Let's not waste time on these. 
# Out of 11k clean tickers, we only saw 1 D and 1 C and those went bankrupt within the year. 
KNOWN_SUFFIXES = {"E", "Q"}

# ── Load ticker list ──────────────────────────────────────────────────────────
tickers_v4 = pd.read_csv(
    DATA_PATH + "../tickers_v4.csv",
    parse_dates=["start_date", "end_date", "start_data", "end_data"],
)
tickers_v4 = tickers_v4[["ticker", "start_date", "end_date"]].copy()

# ── Build trading calendar from the dates in the file ─────────────────────────
# Every date in the file is a valid trading day, so the union of start/end
# dates gives us a complete-enough calendar for the next-day lookup.
all_dates    = pd.concat([tickers_v4["start_date"], tickers_v4["end_date"]]).dropna().unique()
trading_days = sorted(all_dates)

next_trading_day_map = {
    d: trading_days[i + 1]
    for i, d in enumerate(trading_days[:-1])
}

# ── Fast lookup: start_date → set of tickers starting on that day ─────────────
start_date_index = (
    tickers_v4
    .groupby("start_date")["ticker"]
    .apply(set)
    .to_dict()
)

# ── Detect status-suffix renamings ────────────────────────────────────────────
records = []

for row in tickers_v4.itertuples(index=False):   # itertuples: no per-row Series overhead
    old_ticker = row.ticker
    last_date  = row.end_date

    next_day = next_trading_day_map.get(last_date)
    if next_day is None:
        continue

    for new_ticker in start_date_index.get(next_day, set()):
        # Strip the candidate suffix first; check allow-list before the
        # (slightly more expensive) full string comparison.
        if len(new_ticker) != len(old_ticker) + 1:
            continue
        suffix = new_ticker[-1]
        if suffix not in KNOWN_SUFFIXES:          # O(1) set lookup — cheap gate
            continue
        if new_ticker[:-1] != old_ticker:         # exact root match
            continue

        records.append({
            "date":         next_day,
            "ticker_old":   old_ticker,
            "ticker_new":   new_ticker,
            "suffix_added": suffix,
        })

inferred_df = (
    pd.DataFrame(records, columns=["date", "ticker_old", "ticker_new", "suffix_added"])
    .sort_values("date")
    .reset_index(drop=True)
)

print(f"Found {len(inferred_df)} inferred renamings")
print(inferred_df)

if not inferred_df.empty:
    inferred_df.to_csv(PROCESSED / "inferred_renamings.csv", index=False)


Found 0 inferred renamings
Empty DataFrame
Columns: [date, ticker_old, ticker_new, suffix_added]
Index: []


There are a whopping 4347 ticker changes from 2003 to now that we have to take care of. But at least this was very easy to get.

Now we will use them to merge our data. We have to be aware that it is possible for a ticker to used multiple times, so the <code>ticker_changes.csv</code> may contain multiple of the same tickers in the 'from' and 'to' column. 

After processing the ticker changes we will create a <code>tickers_v5.csv</code> which will be our definitive ticker list. This contains a column 'tickers_old', which will containa list of (date_of_change, ticker) pairs. So if A changes to B on day 2, and B changes to C on day 5, tickers_old for D will contain [[2, A], [5, B]].

The process will be as follows:
* As long as we have ticker changes to process
    * Loop through <code>tickers_v4.csv</code>.
        * Get the next trading date after 'end_date_data'.
        * Search in <code>tickers_changes.csv</code> if there is a ticker change on this date.
        * If it does:
            * The stock data will be merged.
            * In <code>tickers_v4.csv</code> we will change "ticker" to the new ticker and add a list [date, ticker] to "tickers_old".
            * All other rows will be merged such as "start_date". For identifiers such as FIGI we will take the last available value. For the ID we will keep the original. If we do not do this, we might run into problems with identical IDs.
            * The row of the old ticker will be deleted
            * **We need to restart the loop!** If we don't the following can happen: Let's assume that a ticker was renamed from A -> B -> C -> D but that the order in which it appears in our ticker list is C, D, A, B. Using our loop, C gets merged with D. Then the loop checks D, which has no renamings. Then A gets merged with B. Then B gets merged with C, however that is incorrect! B should be merged with the new D, which contains C. Any double+ renamings have the risk of being in the 'wrong order'.
                * For this to work, the ticker list must be sorted on end_date.
            * Of course we must not forget that there can be adjustments on day 1 of the ticker change. There should be laws to prohibit this.

Note: if a ticker A goes OTC and then comes back and changes to B, then we will have two files: one of the A before OTC and the A+B after OTC named B. This is as intended.

In [ ]:
tickers_v4 = get_tickers(v=4)
# QUICK BUG FIX, NEED TO REWRITE CODE TO MAKE IT CHRONOLOGICAL
market_dates = get_market_dates()
ticker_changes = get_ticker_changes()

END_DATE = market_dates[-1] # to not get "out of range" errors

# ── Augment ticker_changes with inferred status-suffix renamings ───────────────
# These (e.g. AAII → AAIIE) are not in stockanalysis.com but were inferred
# ── Augment ticker_changes with inferred status-suffix renamings ───────────────
try:
    inferred_path = PROCESSED + "inferred_renamings.csv"
    if os.path.exists(inferred_path):
        inferred = pd.read_csv(inferred_path, parse_dates=["date"])
        inferred["date"] = pd.to_datetime(inferred["date"]).dt.date

        tc_flat  = ticker_changes.reset_index()
        date_col = tc_flat.columns[0]

        inf_flat = (
            inferred[["date", "ticker_old", "ticker_new"]]
            .rename(columns={"date": date_col, "ticker_old": "from", "ticker_new": "to"})
        )

        n_before = len(ticker_changes)
        ticker_changes = (
            pd.concat([tc_flat, inf_flat], ignore_index=True)
            .drop_duplicates(subset=[date_col, "from"], keep="first")
            .set_index(date_col)
            .sort_index()
        )
        print(
            f"ticker_changes augmented: {n_before} → {len(ticker_changes)} entries "
            f"(+{len(inf_flat)} inferred suffix renames)"
        )
except Exception as e:
    print(f"[WARN] Inferred renamings augmentation skipped: {e}")
    print("[WARN] Main loop will proceed with unaugmented ticker_changes")

tickers_v4.insert(loc = 2, column = 'tickers_old', value = [{} for _ in range(len(tickers_v4))])

while True:
    tickers_v4 = tickers_v4.sort_values(by='end_data').reset_index(drop=True)

    # tickers_v4 gets smaller by 1 element every time we run this loop.
    for index_from, row_from in tickers_v4.copy().iterrows():
        # Get values
        type_from = row_from['type']
        if type_from == "INDEX":
            continue
        id_from = row_from['ID']
        ticker_from = row_from['ticker']
        start_date_from = row_from['start_date']
        end_date_from = row_from['end_date']
        start_data_from = row_from['start_data']
        end_data_from = row_from['end_data']

        if end_data_from == END_DATE:
            continue

        # ── DEBUG and safety checks ──────────────────────────────────────────────────────
        if pd.isna(end_data_from):
            print(f"  -> NaT detected! Skipping this ticker.")
            print(f"[WARN] index: {index_from}, ticker: {ticker_from}, "
                        f"end_data_from: {end_data_from} (type: {type(end_data_from)})")
            continue

        start_data_to = market_dates[market_dates.index(end_data_from) + 1]

        # Get ticker changes 
        change = ticker_changes[(ticker_changes.index == start_data_to) & (ticker_changes['from'] == ticker_from)]
        if change.empty:
            continue
        elif len(change) > 1:
            raise Exception("Duplicate!")
        ticker_to = change['to'].values[0]

        # Set values of new ticker
        row_to = tickers_v4[(tickers_v4['start_data'] == start_data_to) & (tickers_v4['ticker'] == ticker_to)]
        if row_to.empty:
            continue
        index_to = row_to.index[0]
        id_to = row_to['ID'].values[0]
        tickers_v4.loc[index_to, "tickers_old"][start_data_to.isoformat()] = ticker_from
        tickers_v4.loc[index_to, "start_date"] = start_date_from
        tickers_v4.loc[index_to, "start_data"] = start_data_from

        # --- IDEMPOTENCY CHECK 1 ---
        if not os.path.exists(DATA_PATH + f"processed/m1/{id_from}.parquet"):
            raise FileNotFoundError(f"[ERROR] {DATA_PATH + f"processed/m1/{id_from}.parquet"} is missing! Expected data for {ticker_from}.")

        # Do the actual merging
        from_ = pd.read_parquet(DATA_PATH + f"processed/m1/{id_from}.parquet")
        to = pd.read_parquet(DATA_PATH + f"processed/m1/{id_to}.parquet")

        # When a ticker changes, the adjustments carry over the the old ticker.
            # Get market close minute to calculate the adjustment factor, and adjust 'to'.
            # (Adjustment factor is the same throughout the day, so market close is arbitrarely chosen)
        calendar = get_market_calendar('datetime')
        start_data_to_dt = calendar.loc[start_data_to, 'regular_close']
        start_data_to_dt_bar = to.loc[start_data_to_dt]
        adjustment_factor = start_data_to_dt_bar['close'] / start_data_to_dt_bar['close_original']
        from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(adjustment_factor)
        from_ = round(from_, 4)

        # Because companies like to be annoying, a split/dividend can take place at the same time as a ticker change. We have to account for this. An example is TYDE -> OCTO with a 50:1 reverse split. However this is much easier than 5_process_raw_data.ipynb, because there is at most a single split. This is rare, but there are still a handful of cases.
        if os.path.isfile(DATA_PATH + f"raw/adjustments/{ticker_to}.csv"):
            adjustments = pd.read_csv(DATA_PATH + f"raw/adjustments/{ticker_to}.csv", parse_dates=True, index_col=0)
            adjustments.index = pd.to_datetime(adjustments.index).date
            adjustments = adjustments[(adjustments.index == start_data_to)]

            # SPLIT ADJUSTMENT
            split = adjustments[adjustments.type == 'SPLIT']
            if len(split) > 0:
                split_amount = split['amount'][0]

                from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(split_amount)

            # DIVIDEND ADJUSTMENT - REUN is the only case, not clear what happened there, likely a 'special dividend'
            dividend = adjustments[adjustments.type == 'DIV']
            if len(dividend) > 0:
                print(ticker_to)
                market_hours = get_market_calendar()
                market_hours = market_hours[['regular_close']]

                cum_div_date = end_data_from
                cum_div_time = market_hours.loc[cum_div_date][0]
                cum_div_datetime = datetime.combine(cum_div_date, cum_div_time)
                cum_div_datetime = (from_[from_.index <= cum_div_datetime].index).max()
                cum_div_close = from_.loc[cum_div_datetime, 'close']
                dividend_amount = dividend['amount'][0]
                    
                adjustment_factor = 1 - dividend_amount/cum_div_close

                from_[['open', 'high', 'low', 'close']] = from_[['open', 'high', 'low', 'close']].multiply(adjustment_factor)
            
            # ROUNDING
            if len(split) > 0 or len(dividend) > 0:
                from_ = round(from_, 4)
                from_['turnover'] = from_['turnover'].astype(int)

        # # If on the old ticker, there are divs/splits on start_data_to (start of new ticker), then something is wrong.
        # if os.path.isfile(DATA_PATH + f"raw/adjustments/{ticker_from}.csv"):
        #     adjustments = pd.read_csv(DATA_PATH + f"raw/adjustments/{ticker_from}.csv", parse_dates=True, index_col=0)
        #     adjustments.index = pd.to_datetime(adjustments.index).date
        #     adjustments = adjustments[(adjustments.index == start_data_to)]
        #     assert len(adjustments) == 0

        # Remove the 'from' ticker, then paste the 'from' and 'to' ticker to m1_renamed for debugging purposes.
        _ = shutil.move(DATA_PATH + f'processed/m1/{id_from}.parquet', DATA_PATH + f'processed/m1_renamed/{id_from}.parquet')
        _ = shutil.copyfile(DATA_PATH + f'processed/m1/{id_to}.parquet', DATA_PATH + f'processed/m1_renamed/{id_to}.parquet')

        pd.concat([from_, to]).to_parquet(DATA_PATH + f"processed/m1/{id_to}.parquet", engine="fastparquet", row_group_offsets=25000)

        tickers_v4.drop(index_from, inplace=True)
        tickers_v4.reset_index(inplace=True, drop=True)
        
        print(f"Ticker change {ticker_from} -> {ticker_to} on {start_data_to} has been processed")
        print(f"{index_from/len(tickers_v4)*100:.1f}% | Length of tickers_v4 is {len(tickers_v4)}")
        break

    # If we have reached the end of the loop, it means we have processed everything. Then we can stop.
    if index_from >= (len(tickers_v4)-1):
        break
    
tickers_v4 = tickers_v4.sort_values(by='ID').reset_index(drop=True)

tickers_v4.to_csv("../data/tickers_v5.csv")

Ticker change GSAH -> VRT on 2020-02-10 has been processed
0.0% | Length of tickers_v4 is 11285
Ticker change SNE -> SONY on 2021-04-01 has been processed
0.0% | Length of tickers_v4 is 11284
Ticker change SRNG -> DNA on 2021-09-17 has been processed
0.0% | Length of tickers_v4 is 11283
Ticker change VIH -> BKKT on 2021-10-18 has been processed
0.0% | Length of tickers_v4 is 11282
Ticker change FB -> META on 2022-06-09 has been processed
0.0% | Length of tickers_v4 is 11281
Ticker change WISH -> LOGC on 2024-05-13 has been processed
0.0% | Length of tickers_v4 is 11280
Ticker change SQ -> XYZ on 2025-01-21 has been processed
0.0% | Length of tickers_v4 is 11279
Ticker change MMC -> MRSH on 2026-01-14 has been processed
0.0% | Length of tickers_v4 is 11278
Ticker change BITF -> KEEL on 2026-04-06 has been processed
0.0% | Length of tickers_v4 is 11277
Ticker change VSCO -> VSXY on 2026-06-02 has been processed
0.0% | Length of tickers_v4 is 11276
  -> NaT detected! Skipping this ticker.

In [13]:
tickers_v5 = get_tickers(v=5)
renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 2] # These were renamed
print(len(renamings))
tickers_v5[tickers_v5['ticker'] == 'META']

10


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi
6405,META-2022-06-09,META,{'2022-06-09': 'FB'},"Meta Platforms, Inc. Class A Common Stock",True,2016-06-06,2026-06-03,2016-06-06,2026-06-03,CS,1326801.0,BBG000MM2P62


Tickers that were renamed multiple times:

In [14]:
multiple_renamings = tickers_v5[tickers_v5["tickers_old"].str.len() > 23]
print(len(multiple_renamings))
multiple_renamings.head(2)

0


,ID,ticker,tickers_old,name,active,start_date,end_date,start_data,end_data,type,cik,composite_figi


Now we have 5 tickers lists. These are:
1. Basic ticker list with a lot of incorrect duplications.
2. Duplications merged and incorrect tickers removed.
3. ETFs added.
4. Data start/end dates added.
5. Renamings merged.
Only the last should be used in backtesting.

If Polygon just provided these from the start, it would have saved countless hours. But at least I learned some Python I guess. And at least Polygon does not ask thousands.

In [15]:
# Check for Negative Prices again! Think I saw one!

# List of tickers with Negative prices
PROCESSED_PATH = "../data/polygon/processed/m1"
FILE_EXT = ".parquet"

for file in os.listdir(PROCESSED_PATH):
    if file.endswith(FILE_EXT):
        ticker = file.replace(FILE_EXT, "")
        file_path = os.path.join(PROCESSED_PATH, file)        
        
        df = pd.read_parquet(file_path)
        
        if (df['close'] < 0).any():
            print(f'{ticker} has negative prices!')

# 8.2 Updates
Rerun the file after setting END_DATE and updating the list of renamings.